# Fall Detection Pipeline (Corrected Dataset Logic)
This notebook assumes:
- falling_01.csv to falling_20.csv => FALL class (1)
- no_falling.csv => NO FALL class (0)


In [ ]:
!pip install -q keras-tuner

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow Version:", tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CHANGE THIS PATH IF NEEDED
DATA_DIR = "/content/drive/MyDrive/FYP/fall_detection"

def process_csi_string(csi_str):

    clean_str = csi_str.replace('[', '').replace(']', '').strip()

    clean_str = ' '.join(clean_str.split())

    csi_data = np.fromstring(
        clean_str,
        sep=' ',
        dtype=int
    )

    # Skip corrupted rows
    if len(csi_data) < 10:
        return None, None

    # Ensure even number of entries
    if len(csi_data) % 2 != 0:
        csi_data = csi_data[:-1]

    imag = csi_data[0::2]
    real = csi_data[1::2]

    amplitude = np.sqrt(imag**2 + real**2)

    phase = np.arctan2(imag, real)

    target_len = 128

    if len(amplitude) < target_len:

        padded_amp = np.zeros(target_len)
        padded_amp[:len(amplitude)] = amplitude
        amplitude = padded_amp

        padded_phase = np.zeros(target_len)
        padded_phase[:len(phase)] = phase
        phase = padded_phase

    elif len(amplitude) > target_len:

        amplitude = amplitude[:target_len]
        phase = phase[:target_len]

    return amplitude, phase

X = np.array(X_list)
X_phase = np.array(X_phase_list)
y = np.array(y_list)

print(f"\nTotal samples loaded: {X.shape[0]}")
print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")

print("\nClass Distribution:")
print("No Fall (0):", np.sum(y == 0))
print("Fall (1):", np.sum(y == 1))

print("\nUnique Labels:", np.unique(y, return_counts=True))

In [ ]:
# Normalize Data

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# CNN Input Shape
X_scaled = X_scaled.reshape(
    X_scaled.shape[0],
    X_scaled.shape[1],
    1
)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
# Build CNN Model

model = Sequential([

    Conv1D(
        32,
        kernel_size=3,
        activation='relu',
        input_shape=(128, 1)
    ),

    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(
        64,
        kernel_size=3,
        activation='relu'
    ),

    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')

])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train Model

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
# Evaluate Model

y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
# Plot Training Curves

plt.figure(figsize=(10,4))

plt.plot(
    history.history['accuracy'],
    label='Train Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")

plt.legend()
plt.show()


plt.figure(figsize=(10,4))

plt.plot(
    history.history['loss'],
    label='Train Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")

plt.legend()
plt.show()

In [ ]:
# Save Keras model

model.save("fall_detection_model.h5")

print("H5 model saved successfully")

In [ ]:
# Convert to TensorFlow Lite

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("fall_detection_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully")